# Data-Preprocessing-Time


In [ ]:
# Reads the consolidated CSV from 1-EDA. Trims by (user, gesture, sensor) using:
#  - one gesture-level window, with sensor-level overrides when available.
#  - A sensor-specific window still takes precedence when defined.
# Saves CSV/XLSX/TXT and generates EDA outputs (PNG + PDF).  

# --- Imports
from pathlib import Path
import sys
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#---------- Reproducibility Configuration
sys.path.insert(0, str(Path("src").resolve()))
from reproducibility import DEFAULT_SEED, configure_reproducibility

SEED = DEFAULT_SEED
configure_reproducibility(SEED, include_tensorflow=False)
print(f"Global reproducibility seed configured: {SEED}")

# --- Paths
BASE_DIR = Path(".")
INPUT_CSV = BASE_DIR / "Data/2-Consolidated/emg_consolidated.csv"

OUT_DIR = BASE_DIR / "Outputs"
PLOTS_DIR = OUT_DIR / "eda_plots_time_clean"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_DIR = BASE_DIR / "Data/2-Consolidated-Time-Clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
CSV_OUT  = CLEAN_DIR / "emg_consolidated_time_clean.csv"
XLSX_OUT = CLEAN_DIR / "emg_consolidated_time_clean.xlsx"
TXT_OUT  = CLEAN_DIR / "emg_consolidated_time_clean.txt"

# --- Sensor disconnected to be removed
SENSORS_TO_DROP = ["EMG Signal (wire) 1"]

# --- Unit of the crop window: "minutes", "seconds" or "auto"
CROP_WINDOW_UNIT = "auto"

# --- If no sensor-level or gesture-level window exists, keep the full segment?
PASSTHROUGH_IF_MISSING = False  # set to True to avoid dropping any group 

# -----------------------------------------------------------------------------
# CONFIG - window by gesture + sensor-specific overrides (optional).
# Priority: sensor-level window when available, then gesture-level window.
# -----------------------------------------------------------------------------
crop_config_v2 = {
    "users": {
        # ====== User 0007 ======
        "0007": {
            "gestures": {
                # One range for the entire gesture, applied to all sensors for that gesture
                "g1": {
                    "window": {"start_min": 6.0, "end_min": 26.0},
                    # To adjust one specific sensor within g1:
                    "sensors": {
                        # "Right Extensor carpi radialis longus": {"start_min": 6.0, "end_min": 26.0}
                    }
                },
                # If "sensors" is omitted, the gesture window is still used for all sensors
                "g2": {"window": {"start_min": 7.0, "end_min": 27.0}},
                "g3": {"window": {"start_min": 3.0, "end_min": 23.0}},
                "g4": {"window": {"start_min": 4.0, "end_min": 24.0}},
                "g5": {"window": {"start_min": 10.0, "end_min": 30.0}},
                "g6": {"window": {"start_min": 5.0, "end_min": 25.0}},
            }
        },

        # ====== User 0008 ======
        "0008": {
            "gestures": {
                "g1": {"window": {"start_min": 4.0, "end_min": 24.0}},
                "g2": {"window": {"start_min": 4.0, "end_min": 24.0}},
                "g3": {"window": {"start_min": 3.0, "end_min": 23.0}},
                "g4": {"window": {"start_min": 5.0, "end_min": 25.0}},
                "g5": {"window": {"start_min": 4.0, "end_min": 24.0}},
                "g6": {"window": {"start_min": 3.0, "end_min": 23.0}},
            }
        },

        # ====== User 0010 ====== 
        "0010": {
            "gestures": {
                "g1": {"window": {"start_min": 4.0, "end_min": 24.0}},
                "g2": {"window": {"start_min": 5.0, "end_min": 25.0}},
                "g3": {"window": {"start_min": 3.0, "end_min": 23.0}},
                "g4": {"window": {"start_min": 5.0, "end_min": 25.0}},
                "g5": {"window": {"start_min": 3.0, "end_min": 23.0}},
                "g6": {"window": {"start_min": 4.0, "end_min": 24.0}},
            }
        },
    }
}

# ==================== Plot helpers ====================
def nice_title(text: str):
    return str(text).replace("_", " ").strip()

def decimate_for_plot(df: pd.DataFrame, max_points: int = 50000) -> pd.DataFrame:
    if len(df) <= max_points: return df
    step = int(np.ceil(len(df) / max_points))
    return df.iloc[::step, :].reset_index(drop=True)

def safe_slug(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(s)).strip("_")

def save_figure_multi(basepath: Path, dpi_png: int = 160):
    plt.tight_layout()
    plt.savefig(basepath.with_name(basepath.name + ".png"), dpi=dpi_png)
    plt.savefig(basepath.with_name(basepath.name + ".pdf"))
    plt.close()

def plot_single_signal(df, time_col, y_col, title, save_base: Path):
    """One plot per signal, with vertical start/end lines and X-axis ticks that include those values."""
    # Exact times of the trimmed segment (no decimation)
    t0 = float(df[time_col].min())
    t1 = float(df[time_col].max())

    # Decimated series for faster plotting
    dfp = decimate_for_plot(df)

    plt.figure(figsize=(14, 4))
    ax = plt.gca()
    ax.plot(dfp[time_col], dfp[y_col], linewidth=1.0)

    # Ensure the X-axis shows the exact segment range
    ax.set_xlim(t0, t1)

    # Vertical start/end lines
    ax.axvline(t0, linestyle="--", alpha=0.5)
    ax.axvline(t1, linestyle="--", alpha=0.5)

    # Force t0 and t1 to appear as X-axis ticks
    xticks = list(ax.get_xticks())
    xticks.extend([t0, t1])
    xticks = sorted(set(xticks))
    ax.set_xticks(xticks)

    # Optional: format ticks with 2 decimals
    ax.set_xticklabels([f"{x:.2f}" for x in xticks])

    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Amplitude [mV]")
    ax.set_title(nice_title(title) + f"  |  x: {t0:.2f} → {t1:.2f} s")

    save_figure_multi(save_base)


def plot_stacked_signals(df, time_col, y_cols, title, save_base: Path):
    """Stacked plot with vertical start/end lines and X-axis ticks that include those values."""
    # Global segment times (no decimation)
    t0 = float(df[time_col].min())
    t1 = float(df[time_col].max())

    # Decimated series for plotting
    dfp = decimate_for_plot(df)

    # Vertical offset calculation for stacking
    amps = []
    for c in y_cols:
        s = dfp[c].values
        q = np.nanpercentile(np.abs(s), 95)
        amps.append(q if q > 0 else np.nanstd(s))
    step = np.nanmedian(amps) * 4.0 if len(amps) else 1.0

    plt.figure(figsize=(16, 9))
    ax = plt.gca()

    for i, c in enumerate(y_cols):
        offset = i * step
        ax.plot(dfp[time_col], dfp[c] + offset, linewidth=0.8, label=c)

    # Exact range and vertical start/end lines
    ax.set_xlim(t0, t1)
    ax.axvline(t0, linestyle="--", alpha=0.5)
    ax.axvline(t1, linestyle="--", alpha=0.5)

    # Force t0 and t1 to appear as X-axis ticks
    xticks = list(ax.get_xticks())
    xticks.extend([t0, t1])
    xticks = sorted(set(xticks))
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{x:.2f}" for x in xticks])

    ax.grid(True, alpha=0.3)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Amplitude + offset [mV]")
    ax.set_title(nice_title(title) + f"  |  x: {t0:.2f} → {t1:.2f} s")
    ax.legend(loc="upper right", fontsize=8, ncol=1, framealpha=0.9)

    save_figure_multi(save_base)

def excel_max_rows_per_sheet(include_header: bool = True) -> int:
    MAX = 1_048_576
    return MAX - (1 if include_header else 0)

# ==================== Trimming helpers ====================
def median_dt_seconds(t: pd.Series) -> float:
    if t.empty: return np.nan
    vals = t.astype(float).values
    dt = np.diff(vals); dt = dt[~np.isnan(dt)]
    if len(dt) == 0: return np.nan
    return float(np.nanmedian(dt))

def to_seconds_group(start_raw: float, end_raw: float, tmin: float, tmax: float) -> tuple[float, float, str]:
    """
    Returns (start_s, end_s, unit_used) according to:
      - "minutes": strictly minutes-to-seconds.
      - "seconds": usa tal cual.
      - "auto": tries minutes; if it does not fit and the full segment is short, uses seconds.
    """
    unit = CROP_WINDOW_UNIT.lower().strip()
    m_start, m_end = start_raw * 60.0, end_raw * 60.0
    s_start, s_end = start_raw, end_raw

    if unit == "minutes":
        return m_start, m_end, "minutes"
    if unit == "seconds":
        return s_start, s_end, "seconds"

    # AUTO
    if (m_start >= tmin) and (m_end <= tmax):
        return m_start, m_end, "minutes"
    if (tmax - tmin) <= 120.0 or (m_end > tmax and (m_end - tmin) > 2.0 * (tmax - tmin)):
        if (s_start >= tmin) and (s_end <= tmax):
            return s_start, s_end, "seconds"

    inter_m = max(0.0, min(m_end, tmax) - max(m_start, tmin))
    inter_s = max(0.0, min(s_end, tmax) - max(s_start, tmin))
    if inter_s >= inter_m and inter_s > 0.0:
        return s_start, s_end, "seconds"
    return m_start, m_end, "minutes"

def flatten_v2(cfg: dict):
    """
    Without normalizing keys.
    Retorna dos mapas:
      - sensor_map:  (u,g,s) -> (start_raw, end_raw)  [prioridad 1]
      - gesture_map: (u,g)   -> (start_raw, end_raw)  [prioridad 2]
    """
    sensor_map, gesture_map = {}, {}
    users_blk = (cfg or {}).get("users", {}) if isinstance(cfg, dict) else {}
    for u_key, u_blk in users_blk.items():
        g_blk = (u_blk or {}).get("gestures", {})
        for g_key, gcfg in g_blk.items():
            # gesture-level window
            gwin = (gcfg or {}).get("window", None)
            if isinstance(gwin, dict) and ("start_min" in gwin) and ("end_min" in gwin):
                sraw = float(gwin["start_min"]); eraw = float(gwin["end_min"])
                if eraw > sraw:
                    gesture_map[(str(u_key), str(g_key))] = (sraw, eraw)
            # overrides por sensor (si existen)
            s_blk = (gcfg or {}).get("sensors", {})
            for s_key, scfg in s_blk.items():
                if not isinstance(scfg, dict): 
                    continue
                if ("start_min" not in scfg) or ("end_min" not in scfg): 
                    continue
                sraw = float(scfg["start_min"]); eraw = float(scfg["end_min"])
                if eraw > sraw:
                    sensor_map[(str(u_key), str(g_key), str(s_key))] = (sraw, eraw)
    return sensor_map, gesture_map

# ==================== CSV loading ====================
big_df = pd.read_csv(INPUT_CSV, dtype={"user": str, "gesture": str})

# Validations
for col in ["user", "gesture", "Time"]:
    if col not in big_df.columns:
        raise ValueError(f"Required column '{col}' not found in {INPUT_CSV}")

# Eliminar sensor desconectado
to_drop = [c for c in SENSORS_TO_DROP if c in big_df.columns]
if to_drop:
    big_df = big_df.drop(columns=to_drop)

# Ensure numeric Time and sort
big_df["Time"] = pd.to_numeric(big_df["Time"], errors="coerce")
big_df = big_df.dropna(subset=["Time"]).sort_values(["user", "gesture", "Time"]).reset_index(drop=True)

# Sensors = all columns except id/time/frame
id_cols   = ["user", "gesture"]
frame_col = "Frame" if "Frame" in big_df.columns else None
non_signal = set(id_cols + ["Time"] + ([frame_col] if frame_col else []))
data_sensors = [c for c in big_df.columns if c not in non_signal]

# Info
users_present    = big_df["user"].unique().tolist()
gestures_present = big_df["gesture"].unique().tolist()
print("Users in CSV:", users_present)
print("Gestures in CSV:", gestures_present)
print("Sensors in CSV:", data_sensors)
print("Crop unit:", CROP_WINDOW_UNIT, "| PASSTHROUGH_IF_MISSING:", PASSTHROUGH_IF_MISSING)

# Indexes from config v2
sensor_map, gesture_map = flatten_v2(crop_config_v2)

# Long/tidy
melt_id = id_cols + (["Frame"] if frame_col else []) + ["Time"]
long_df = big_df.melt(id_vars=melt_id, value_vars=data_sensors,
                      var_name="sensor", value_name="value")
long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")

# ==================== Apply trims with gesture-to-sensor fallback ====================
clean_parts, skips, notes = [], [], []
auto_seconds_used = 0
minutes_used = 0
used_sensor_level = 0
used_gesture_level = 0
passthrough_count = 0

for (u, g, s), grp in long_df.groupby(["user", "gesture", "sensor"], sort=False):
    grp = grp.sort_values("Time").reset_index(drop=True)
    key_s = (str(u), str(g), str(s))
    key_g = (str(u), str(g))
    tmin, tmax = float(grp["Time"].min()), float(grp["Time"].max())

    # 1) priority: sensor-level window
    if key_s in sensor_map:
        start_raw, end_raw = sensor_map[key_s]
        used_level = "sensor"
        used_sensor_level += 1
    # 2) fallback: gesture-level window
    elif key_g in gesture_map:
        start_raw, end_raw = gesture_map[key_g]
        used_level = "gesture"
        used_gesture_level += 1
    # 3) if nothing is configured
    else:
        if PASSTHROUGH_IF_MISSING:
            sel = grp.copy()
            clean_parts.append(sel)
            passthrough_count += 1
        else:
            skips.append((u, g, s, "no window (sensor nor gesture)"))
        continue

    # convert to seconds according to "auto"/"minutes"/"seconds"
    start_s, end_s, used_unit = to_seconds_group(start_raw, end_raw, tmin, tmax)
    if used_unit == "seconds": auto_seconds_used += 1
    if used_unit == "minutes": minutes_used += 1

    # if the calculated window does not fit, adjust to the valid range to keep the full group
    left  = max(start_s, tmin)
    right = min(end_s,   tmax)
    if right <= left:
        notes.append((u, g, s, f"window out of bounds [{tmin:.2f},{tmax:.2f}] vs [{start_s:.2f},{end_s:.2f}] ({used_level}/{used_unit})"))
        continue

    sel = grp[(grp["Time"] >= left) & (grp["Time"] <= right)].copy()

    dt = median_dt_seconds(sel["Time"])
    if np.isnan(dt) or len(sel) < 2:
        notes.append((u, g, s, f"empty/invalid selection ({used_level}/{used_unit})"))
        continue

    # ajuste suave
    measured = float(sel["Time"].iloc[-1] - sel["Time"].iloc[0])
    target   = float(right - left)
    if abs(measured - target) > (2.0 * dt):
        sel = sel[sel["Time"] <= (sel["Time"].iloc[0] + target + 1e-9)]

    clean_parts.append(sel)

# ==================== Save outputs ====================
save_cols = ["user", "gesture"] + (["Frame"] if frame_col else []) + ["Time", "sensor", "value"]

if not clean_parts:
    print("\n=== NO MATCHES CROPPED ===")
    pd.DataFrame(columns=save_cols).to_csv(CSV_OUT, index=False)
    pd.DataFrame(columns=save_cols).to_csv(TXT_OUT, index=False, sep="\t")
    with pd.ExcelWriter(XLSX_OUT) as w:
        pd.DataFrame(columns=save_cols).to_excel(w, index=False, sheet_name="All")
    print(f"Empty CSV  -> {CSV_OUT.resolve()}")
    print(f"Empty XLSX -> {XLSX_OUT.resolve()}")
    print(f"Empty TXT  -> {TXT_OUT.resolve()}")
    if skips:
        print("\nSkipped groups (first 12):")
        for row in skips[:12]:
            print("  user=", row[0], "| gesture=", row[1], "| sensor=", row[2], "=>", row[3])
        if len(skips) > 12: print(f"  ... and {len(skips)-12} more.")
    if notes:
        print("\nNotes (first 12):")
        for row in notes[:12]:
            print("  user=", row[0], "| gesture=", row[1], "| sensor=", row[2], "=>", row[3])
        if len(notes) > 12: print(f"  ... and {len(notes)-12} more.")
else:
    clean_long = pd.concat(clean_parts, ignore_index=True)[save_cols]
    clean_long.to_csv(CSV_OUT, index=False)
    clean_long.to_csv(TXT_OUT, index=False, sep="\t")

    max_rows_excel = excel_max_rows_per_sheet(include_header=True)
    with pd.ExcelWriter(XLSX_OUT) as writer:
        if len(clean_long) <= max_rows_excel:
            clean_long.to_excel(writer, index=False, sheet_name="All")
        else:
            start = 0; part = 1
            while start < len(clean_long):
                end = min(start + max_rows_excel, len(clean_long))
                clean_long.iloc[start:end].to_excel(writer, index=False, sheet_name=f"Part_{part}")
                start = end; part += 1

    print(f"\nSaved CSV  -> {CSV_OUT.resolve()}")
    print(f"Saved XLSX -> {XLSX_OUT.resolve()}")
    print(f"Saved TXT  -> {TXT_OUT.resolve()}")
    print(f"Used windows -> sensor-level: {used_sensor_level} | gesture-level: {used_gesture_level} | passthrough: {passthrough_count}")
    print(f"Units used (auto) -> seconds: {auto_seconds_used} | minutes: {minutes_used}")

    # ==================== Plots EDA (time-clean) ====================
    # 1) Per signal
    for (u, g, s), grp in clean_long.groupby(["user", "gesture", "sensor"], sort=False):
        base = PLOTS_DIR / f"user_{u}" / f"{g}"
        base.mkdir(parents=True, exist_ok=True)
        title = f"User {u} | {g.upper()} | Signal: {s} (time-clean)"
        save_base = base / f"{g}_{safe_slug(s)}"
        plot_single_signal(grp.rename(columns={"value": s}), "Time", s, title, save_base)

    # 2) All signals stacked by (user, gesture)
    for (u, g), grp in clean_long.groupby(["user", "gesture"], sort=False):
        rng = grp.groupby("sensor").agg(start=("Time", "min"), end=("Time", "max")).reset_index()
        start_s = float(rng["start"].max()); end_s = float(rng["end"].min())
        if end_s <= start_s: continue
        sub = grp[(grp["Time"] >= start_s) & (grp["Time"] <= end_s)].copy()
        wide = sub.pivot_table(index="Time", columns="sensor", values="value", aggfunc="first").reset_index()
        wide = wide.sort_values("Time").reset_index(drop=True)
        sig_cols = [c for c in wide.columns if c != "Time"]
        if len(sig_cols) < 2: continue
        base = PLOTS_DIR / f"user_{u}" / f"{g}"
        base.mkdir(parents=True, exist_ok=True)
        title = f"User {u} | {g.upper()} | All signals (stacked, time-clean)"
        save_base = base / f"{g}_ALL_SIGNALS"
        plot_stacked_signals(wide, "Time", sig_cols, title, save_base)

    print(f"\nTime-clean EDA plots -> {PLOTS_DIR.resolve()}")

    # ==================== Quick summary ====================
    summary = (
        clean_long.groupby(["user", "gesture", "sensor"], as_index=False)
                  .agg(start_time_s=("Time", "min"),
                       end_time_s=("Time", "max"),
                       n_rows=("Time", "count"))
    )
    summary["duration_s"] = summary["end_time_s"] - summary["start_time_s"]
    summary = summary.sort_values(["user", "gesture", "sensor"]).reset_index(drop=True)
    try:
        display(summary.head(20))
    except Exception:
        print("\nSummary preview (top 20 rows):")
        print(summary.head(20).to_string(index=False))
# =========================================================================================
